In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_5.py ──
"""
Shared infrastructure for MLFP03 Exercise 5 — Class Imbalance & Calibration.

Contains: Singapore credit scoring data loader, Kailash PreprocessingPipeline
wiring, cost-matrix constants, metric helpers, reliability-diagram helpers,
and an OUTPUT_DIR convention for every technique file to write visual proof
to the same place.

Technique-specific code (SMOTE call, focal loss gradient, Platt/Isotonic
model wiring) does NOT belong here — it lives in the per-technique files.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY — every technique writes visual proof to the same place
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex5_imbalance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# BUSINESS CONTEXT — Singapore retail bank credit scoring
# ════════════════════════════════════════════════════════════════════════
# These constants drive every technique file. A 100:1 cost ratio is
# realistic for SEA consumer lending: the average charged-off unsecured
# loan in Singapore is ~S$10,000 (MAS consumer credit report 2024), and
# the operational cost of a false decline (manual review + lost NPV of
# the customer relationship) is roughly S$100.


@dataclass(frozen=True)
class CostMatrix:
    """Dollar cost of each confusion-matrix cell.

    fn = cost of missing a default (charge-off loss)
    fp = cost of a false alarm (manual review + lost relationship NPV)
    """

    fn: float = 10_000.0
    fp: float = 100.0

    @property
    def optimal_threshold(self) -> float:
        """Bayes-optimal threshold for this cost matrix: t* = fp / (fp + fn)."""
        return self.fp / (self.fp + self.fn)

    def total_cost(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        return float(fp * self.fp + fn * self.fn)


DEFAULT_COSTS = CostMatrix(fn=10_000.0, fp=100.0)

# Annual volume for ROI analysis — calibrated to a mid-tier SG retail bank.
ANNUAL_APPLICATIONS = 100_000


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring dataset
# ════════════════════════════════════════════════════════════════════════
# The dataset is loaded through the MLFPDataLoader so it works identically
# in local (.data_cache) and Colab (Drive + gdown) formats.


def load_credit_splits(
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Load the SG credit scoring dataset and return (X_train, y_train, X_test, y_test, pos_rate).

    Uses kailash-ml PreprocessingPipeline for consistent preprocessing across
    every technique file. Returns numpy arrays ready for sklearn-style fit.
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target="default",
        seed=seed,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_cols = [c for c in result.train_data.columns if c != "default"]
    X_train, y_train, _ = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    pos_rate = float(y_train.mean())
    return X_train, y_train, X_test, y_test, pos_rate


# ════════════════════════════════════════════════════════════════════════
# METRIC HELPERS — complete taxonomy in one call
# ════════════════════════════════════════════════════════════════════════


def metrics_row(
    name: str,
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, Any]:
    """Compute a full metrics row for a given strategy name + probabilities.

    Returns a dict compatible with polars DataFrame construction so every
    technique file can build comparison tables with identical column shapes.
    """
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        "strategy": name,
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def print_metrics_table(rows: list[dict[str, Any]], title: str) -> None:
    """Print a comparison table across strategies."""
    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}")
    print(
        f"{'Strategy':<22} {'AUC-PR':>8} {'Brier':>8} {'F1':>8} "
        f"{'Precision':>10} {'Recall':>8}"
    )
    print("─" * 70)
    for r in rows:
        print(
            f"{r['strategy']:<22} {r['auc_pr']:>8.4f} {r['brier']:>8.4f} "
            f"{r['f1']:>8.4f} {r['precision']:>10.4f} {r['recall']:>8.4f}"
        )


def rows_to_dataframe(rows: list[dict[str, Any]]) -> pl.DataFrame:
    """Convert metrics rows to a polars DataFrame (for persistence / plots)."""
    return pl.DataFrame(rows)


# ════════════════════════════════════════════════════════════════════════
# RELIABILITY DIAGRAM DATA (calibration curve, binned)
# ════════════════════════════════════════════════════════════════════════


def reliability_bins(
    y_true: np.ndarray, y_proba: np.ndarray, n_bins: int = 10
) -> pl.DataFrame:
    """Compute a binned reliability diagram as a polars DataFrame.

    Columns: bin_lower, bin_upper, mean_pred, empirical_rate, count, gap
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (y_proba >= lo) & (y_proba < hi if i < n_bins - 1 else y_proba <= hi)
        count = int(mask.sum())
        if count == 0:
            continue
        mean_pred = float(y_proba[mask].mean())
        empirical = float(y_true[mask].mean())
        rows.append(
            {
                "bin_lower": float(lo),
                "bin_upper": float(hi),
                "mean_pred": mean_pred,
                "empirical_rate": empirical,
                "count": count,
                "gap": float(abs(mean_pred - empirical)),
            }
        )
    return pl.DataFrame(rows)


def print_reliability(name: str, bins: pl.DataFrame) -> None:
    print(f"\n  Reliability bins — {name}")
    print(f"  {'mean_pred':>10} {'empirical':>10} {'|gap|':>8} {'n':>6}")
    print("  " + "─" * 38)
    for row in bins.iter_rows(named=True):
        print(
            f"  {row['mean_pred']:>10.3f} {row['empirical_rate']:>10.3f} "
            f"{row['gap']:>8.3f} {row['count']:>6}"
        )


# ════════════════════════════════════════════════════════════════════════
# BUSINESS ROI CALCULATOR
# ════════════════════════════════════════════════════════════════════════


def annual_roi(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float,
    costs: CostMatrix = DEFAULT_COSTS,
    annual_volume: int = ANNUAL_APPLICATIONS,
) -> dict[str, float]:
    """Project test-set confusion matrix onto an annual volume.

    Returns a dict with caught_defaults, missed_defaults, false_alarms,
    model_cost, no_model_cost, annual_savings — all in dollars.
    """
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    pos_rate = float(y_true.mean())
    scale = annual_volume / len(y_true)
    annual_fn = fn * scale
    annual_fp = fp * scale
    annual_tp = tp * scale
    n_defaults_annual = pos_rate * annual_volume
    model_cost = annual_fn * costs.fn + annual_fp * costs.fp
    no_model_cost = n_defaults_annual * costs.fn
    return {
        "threshold": float(threshold),
        "defaults_caught": float(annual_tp),
        "defaults_missed": float(annual_fn),
        "false_alarms": float(annual_fp),
        "model_cost_usd": float(model_cost),
        "no_model_cost_usd": float(no_model_cost),
        "annual_savings_usd": float(no_model_cost - model_cost),
        "annual_volume": int(annual_volume),
    }


def print_roi(name: str, roi: dict[str, float]) -> None:
    print(f"\n  ROI — {name}")
    print(f"    Threshold:          {roi['threshold']:.4f}")
    print(f"    Defaults caught:    {roi['defaults_caught']:>12,.0f}")
    print(f"    Defaults missed:    {roi['defaults_missed']:>12,.0f}")
    print(f"    False alarms:       {roi['false_alarms']:>12,.0f}")
    print(f"    Model cost:         ${roi['model_cost_usd']:>12,.0f}")
    print(f"    No-model cost:      ${roi['no_model_cost_usd']:>12,.0f}")
    print(f"    Annual savings:     ${roi['annual_savings_usd']:>12,.0f}")


# ════════════════════════════════════════════════════════════════════════
# PERSISTENCE — shared parquet of per-strategy probabilities
# ════════════════════════════════════════════════════════════════════════
# Each technique file writes its y_proba vector to a parquet under
# OUTPUT_DIR so that later technique files (threshold opt, calibration,
# final comparison) can read them without re-training.

PROBA_STORE = OUTPUT_DIR / "strategy_probabilities.parquet"


def save_strategy_proba(name: str, y_proba: np.ndarray) -> None:
    """Upsert a strategy's probability vector into the shared parquet store."""
    new_df = pl.DataFrame({"strategy": [name] * len(y_proba), "y_proba": y_proba})
    if PROBA_STORE.exists():
        existing = pl.read_parquet(PROBA_STORE)
        existing = existing.filter(pl.col("strategy") != name)
        combined = pl.concat([existing, new_df])
    else:
        combined = new_df
    combined.write_parquet(PROBA_STORE)


def load_strategy_proba(name: str) -> np.ndarray:
    """Read back a previously-saved probability vector for a strategy."""
    if not PROBA_STORE.exists():
        raise FileNotFoundError(
            f"{PROBA_STORE} not found — run the earlier technique files first."
        )
    df = pl.read_parquet(PROBA_STORE).filter(pl.col("strategy") == name)
    if df.height == 0:
        raise KeyError(f"Strategy {name!r} not found in {PROBA_STORE}")
    return df["y_proba"].to_numpy()


def list_saved_strategies() -> list[str]:
    if not PROBA_STORE.exists():
        return []
    df = pl.read_parquet(PROBA_STORE)
    return df["strategy"].unique().to_list()




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 5.4: Threshold Optimisation from a Cost Matrix
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why threshold=0.5 is almost always wrong for asymmetric costs
#   - The Bayes-optimal threshold formula t* = cost_FP / (cost_FP + cost_FN)
#   - How to sweep the threshold empirically and find the cost minimum
#   - How to translate the chosen threshold into annual S$ savings
#
# PREREQUISITES: 02_sampling_strategies.py (cost-sensitive proba saved)
# ESTIMATED TIME: ~25 min
#
# 5-PHASE STRUCTURE:
#   Theory   — decision theory for asymmetric costs
#   Build    — load saved cost-sensitive probabilities, build threshold grid
#   Train    — (no training — we TUNE a decision rule on fixed probabilities)
#   Visualise — cost vs threshold curve + confusion matrix at best threshold
#   Apply    — Maybank Singapore unsecured-loan underwriting ROI
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv


load_dotenv()



## THEORY — Decision theory for asymmetric cost matrices


In [ ]:
# Imagine your model outputs a probability p that an applicant will
# default. You must DECIDE: approve or decline. The default rule is
# "decline if p >= 0.5". Where does 0.5 come from?
#
# It comes from an implicit assumption: the cost of a false positive
# equals the cost of a false negative. In credit scoring, FP costs
# ~S$100 (manual review) and FN costs ~S$10,000 (charge-off). The
# ratio is 100:1. The 0.5 threshold is wrong by two orders of
# magnitude.
#
# THE BAYES-OPTIMAL THRESHOLD:
#
#     t* = cost_FP / (cost_FP + cost_FN)
#
# Expected cost at threshold t:
#     E[cost] = P(y=0) * P(pred=1 | y=0) * cost_FP
#             + P(y=1) * P(pred=0 | y=1) * cost_FN
#
# Minimising over t gives t* above. For our 100:1 cost matrix,
# t* = 100 / 10100 = 0.0099 — roughly 1%. Any applicant with >=1%
# default probability should be declined.
#
# CAVEAT: the theoretical formula assumes the probabilities are
# well-calibrated. If the model outputs scores that are NOT true
# probabilities (most gradient boosters), you should ALSO sweep
# the threshold empirically on a validation set and pick the
# actual argmin of total cost. Exercise 5.5 will fix the calibration
# so the theoretical formula works; this file uses the empirical
# sweep so it's robust to miscalibrated inputs.



## BUILD — load saved probabilities, sweep thresholds


In [ ]:
X_train, y_train, X_test, y_test, pos_rate = load_credit_splits()

try:
    y_proba = load_strategy_proba("cost_sensitive_scale")
except FileNotFoundError as e:
    raise RuntimeError(
        "Run 02_sampling_strategies.py first — it saves the probabilities."
    ) from e

print("\n" + "=" * 70)
print("  Exercise 5.4 — Threshold Optimisation")
print("=" * 70)
print(f"  Cost matrix: FP=${DEFAULT_COSTS.fp:,.0f}, FN=${DEFAULT_COSTS.fn:,.0f}")
print(f"  Bayes-optimal theoretical threshold: {DEFAULT_COSTS.optimal_threshold:.4f}")



## TRAIN (no training — we TUNE a decision rule on fixed probabilities)


In [ ]:
thresholds = np.arange(0.005, 0.500, 0.005)
sweep_rows: list[dict] = []
for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    tp = int(((y_pred == 1) & (y_test == 1)).sum())
    fp = int(((y_pred == 1) & (y_test == 0)).sum())
    fn = int(((y_pred == 0) & (y_test == 1)).sum())
    tn = int(((y_pred == 0) & (y_test == 0)).sum())
    cost = fp * DEFAULT_COSTS.fp + fn * DEFAULT_COSTS.fn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    sweep_rows.append(
        {
            "threshold": float(t),
            "total_cost_usd": float(cost),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
            "precision": float(precision),
            "recall": float(recall),
        }
    )

sweep_df = pl.DataFrame(sweep_rows)
best_row = min(sweep_rows, key=lambda r: r["total_cost_usd"])
best_t = best_row["threshold"]
default_cost = (
    next(r["total_cost_usd"] for r in sweep_rows if abs(r["threshold"] - 0.5) < 1e-6)
    if any(abs(r["threshold"] - 0.5) < 1e-6 for r in sweep_rows)
    else (((y_proba >= 0.5).astype(int) != y_test).sum() * DEFAULT_COSTS.fn)
)
# Always compute the t=0.5 reference cost directly for robustness
y_pred_default = (y_proba >= 0.5).astype(int)
fp_d = int(((y_pred_default == 1) & (y_test == 0)).sum())
fn_d = int(((y_pred_default == 0) & (y_test == 1)).sum())
default_cost = fp_d * DEFAULT_COSTS.fp + fn_d * DEFAULT_COSTS.fn



### Checkpoint 4


In [ ]:
assert 0.0 < best_t < 1.0, "Best threshold must be in (0,1)"
assert (
    best_row["total_cost_usd"] <= default_cost
), "Optimised threshold should not cost more than t=0.5"
print("[ok] Checkpoint 4 — threshold sweep and argmin computed\n")



## VISUALISE — cost curve + confusion matrix at best threshold


In [ ]:
print(
    f"\n  {'threshold':>10} {'total_cost':>14} {'precision':>10} "
    f"{'recall':>8} {'fp':>6} {'fn':>6}"
)
print("  " + "─" * 60)
for r in sweep_rows[::10]:
    print(
        f"  {r['threshold']:>10.3f} ${r['total_cost_usd']:>12,.0f} "
        f"{r['precision']:>10.4f} {r['recall']:>8.4f} {r['fp']:>6} {r['fn']:>6}"
    )

print(f"\n  Cost at t=0.5:       ${default_cost:,.0f}")
print(f"  Cost at t*={best_t:.3f}: ${best_row['total_cost_usd']:,.0f}")
savings = default_cost - best_row["total_cost_usd"]
saving_pct = savings / max(default_cost, 1)
print(f"  Savings from threshold tuning: ${savings:,.0f} ({saving_pct:.1%})")
print(
    f"  Theoretical t* (Bayes): {DEFAULT_COSTS.optimal_threshold:.4f} — "
    f"empirical: {best_t:.4f}"
)

sweep_df.write_parquet(OUTPUT_DIR / "threshold_sweep.parquet")
print(f"\n  Saved: {OUTPUT_DIR / 'threshold_sweep.parquet'}")



### Visual: Threshold vs metrics curve


In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[r["threshold"] for r in sweep_rows],
        y=[r["total_cost_usd"] for r in sweep_rows],
        mode="lines",
        name="Total cost ($)",
        line=dict(color="#ef4444", width=3),
    )
)
fig.add_vline(
    x=best_t, line_dash="dash", line_color="#10b981", annotation_text=f"t*={best_t:.3f}"
)
fig.add_vline(
    x=0.5, line_dash="dot", line_color="#6b7280", annotation_text="t=0.5 (default)"
)
fig.update_layout(
    title="Threshold vs Total Cost: the U-shaped curve (lower = better)",
    xaxis_title="Decision threshold",
    yaxis_title="Total cost ($)",
    height=450,
)
viz_path = OUTPUT_DIR / "ex5_04_threshold_cost_curve.html"
fig.write_html(str(viz_path))
print(f"  Saved: {viz_path}")



### Visual: Precision-Recall vs threshold


In [ ]:
fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=[r["threshold"] for r in sweep_rows],
        y=[r["precision"] for r in sweep_rows],
        mode="lines",
        name="Precision",
        line=dict(color="#6366f1", width=2),
    )
)
fig2.add_trace(
    go.Scatter(
        x=[r["threshold"] for r in sweep_rows],
        y=[r["recall"] for r in sweep_rows],
        mode="lines",
        name="Recall",
        line=dict(color="#f59e0b", width=2),
    )
)
fig2.add_vline(
    x=best_t, line_dash="dash", line_color="#10b981", annotation_text=f"t*={best_t:.3f}"
)
fig2.update_layout(
    title="Precision and Recall vs Threshold (credit default)",
    xaxis_title="Decision threshold",
    yaxis_title="Score",
    height=450,
    legend=dict(orientation="h", y=-0.2),
)
viz_path2 = OUTPUT_DIR / "ex5_04_threshold_pr_curve.html"
fig2.write_html(str(viz_path2))
print(f"  Saved: {viz_path2}")



### Visual: Cost matrix heatmap


In [ ]:
cost_matrix = np.array([[0, DEFAULT_COSTS.fp], [DEFAULT_COSTS.fn, 0]])
fig3 = go.Figure(
    data=go.Heatmap(
        z=cost_matrix,
        x=["Predicted Negative", "Predicted Positive"],
        y=["Actual Negative", "Actual Positive"],
        text=[
            [f"TN: $0", f"FP: ${DEFAULT_COSTS.fp:,.0f}"],
            [f"FN: ${DEFAULT_COSTS.fn:,.0f}", f"TP: $0"],
        ],
        texttemplate="%{text}",
        colorscale="Reds",
        showscale=True,
    )
)
fig3.update_layout(
    title="Cost Matrix: Asymmetric penalties (FN >> FP)",
    height=400,
)
viz_path3 = OUTPUT_DIR / "ex5_04_cost_matrix_heatmap.html"
fig3.write_html(str(viz_path3))
print(f"  Saved: {viz_path3}")

# Per-strategy metric table at the optimised threshold
row = metrics_row("Cost-sens @ t*", y_test, y_proba, threshold=best_t)
print_metrics_table([row], f"Metrics at optimised threshold t*={best_t:.3f}")

# INTERPRETATION: The cost curve is U-shaped. At t=0.5 the model misses
# most defaults (FN term dominates). At t=0.01 the model flags everyone
# (FP term dominates). The minimum is far below 0.5 because the cost
# matrix is asymmetric.



## APPLY — Maybank Singapore unsecured-loan underwriting ROI


In [ ]:
# SCENARIO: Maybank Singapore underwrites ~100,000 unsecured personal
# loans per year. Today's rule: decline if scorecard p >= 0.5. The
# underwriting committee asks: "What would it save if we tuned the
# threshold from the cost matrix?"
#
# We can answer that question NOW by scaling the test-set confusion
# matrix to the annual application volume.

roi_default = annual_roi(
    y_test, y_proba, threshold=0.5, annual_volume=ANNUAL_APPLICATIONS
)
roi_best = annual_roi(
    y_test, y_proba, threshold=best_t, annual_volume=ANNUAL_APPLICATIONS
)

print_roi("Annual ROI @ t=0.5", roi_default)
print_roi(f"Annual ROI @ t*={best_t:.3f}", roi_best)

delta = roi_best["annual_savings_usd"] - roi_default["annual_savings_usd"]
print(f"\n  Threshold tuning alone adds: ${delta:,.0f}/year in savings")
print("    No retraining. No new data. No new features. Just changing a")
print("    number in the decision layer. This is usually the highest-ROI")
print("    lever on any imbalanced production model.")

# Persist the annual ROI snapshot for the final comparison in 05
pl.DataFrame(
    [roi_default | {"label": "t=0.5"}, roi_best | {"label": "t*"}]
).write_parquet(OUTPUT_DIR / "threshold_roi.parquet")



## REFLECTION


[x] Derived the Bayes-optimal threshold t* = cost_FP/(cost_FP+cost_FN)
  [x] Ran an empirical sweep to find the argmin of total cost
  [x] Observed the U-shape of the cost-vs-threshold curve
  [x] Scaled the test confusion matrix to annual application volume
  [x] Quantified the dollar savings of threshold tuning alone

  KEY INSIGHT: Threshold tuning is the highest-ROI lever on any
  imbalanced production model. It's free (no retraining) and can add
  seven figures of annual savings on its own.

  Next: 05_calibration.py — Platt and Isotonic calibration make the
  theoretical t* match the empirical t*, and compare every strategy.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — 5.4")
print("=" * 70)
print(
)

